In [1]:
import pandas as pd
from datetime import datetime
from chinese_calendar import is_workday
from spatial_flow_clustering_fuc import init_bike_record_with_sfc_obj, get_near_record_uuid_list, \
    calculate_spatial_dissimilarity, \
    plot_sfc_obj
from spatial_temporal_flow_clustering_fuc import SpatialTemporalFlowCluster

In [2]:
# read sample_bike_records.csv
sample_bike_records_df = pd.read_csv('data\sample_bike_records.csv')
# add is_weekday field
sample_bike_records_df['is_weekday'] = pd.to_datetime(sample_bike_records_df['date']).apply(is_workday)
# get the uid for each user in sample_bike_records.csv
uid_list = sample_bike_records_df['uid'].unique()
# calculate the number of activity weekday for each user
activity_weekdays_dict = {}
for uid in uid_list:
    each_user_weekday_record_df = sample_bike_records_df.query(f'uid == "{uid}" and is_weekday == True')
    activity_weekdays_dict[uid] = len(each_user_weekday_record_df['date'].unique())
activity_weekdays_dict

{'945a5b919ea73a83eeee7c**********': 35}

In [3]:
# get weekday bike record dict
each_spatial_flow_cluster_dict = {}
for uid in uid_list:
    each_user_weekday_record_df = sample_bike_records_df.query(f'uid == "{uid}" and is_weekday == True')
    each_user_weekday_dict = each_user_weekday_record_df.to_dict(orient='records')
    bike_record_dict, init_bike_record_with_sfc_dict = init_bike_record_with_sfc_obj(each_user_weekday_dict)
    bike_record_uuid_list = bike_record_dict.keys()

    for uuid in bike_record_uuid_list:
        this_spatial_flow_cluster = init_bike_record_with_sfc_dict[uuid]
        near_record_uuid_list = get_near_record_uuid_list(bike_record_dict, uuid, this_spatial_flow_cluster.sfc_id)
        for near_uuid in near_record_uuid_list:
            near_spatial_flow_cluster = init_bike_record_with_sfc_dict[near_uuid]
            if this_spatial_flow_cluster.sfc_id != near_spatial_flow_cluster.sfc_id:
                flows_sd = calculate_spatial_dissimilarity(
                    this_spatial_flow_cluster, near_spatial_flow_cluster, _size_coefficient=0.3, _max_circle_boundary_radius=200)
                if flows_sd <= 1:
                    this_spatial_flow_cluster.add_flow(
                        near_spatial_flow_cluster.flow,
                        near_spatial_flow_cluster.including_record_detail)
                    for including_record_uuid in near_spatial_flow_cluster.including_record_detail.keys():
                        init_bike_record_with_sfc_dict[including_record_uuid] = this_spatial_flow_cluster
                        bike_record_dict[including_record_uuid]['sfc_id'] = this_spatial_flow_cluster.sfc_id

    final_spatial_flow_cluster_dict = {}
    min_sfc_threshold = activity_weekdays_dict[uid] / 5
    for uuid, sfc_obj in init_bike_record_with_sfc_dict.items():
        this_sfc_id = sfc_obj.sfc_id
        if this_sfc_id not in final_spatial_flow_cluster_dict.keys() and sfc_obj.record_num >= min_sfc_threshold:
            final_spatial_flow_cluster_dict[this_sfc_id] = sfc_obj
    each_spatial_flow_cluster_dict[uid] = final_spatial_flow_cluster_dict
each_spatial_flow_cluster_dict

{'945a5b919ea73a83eeee7c**********': {'sfc000': <spatial_flow_clustering_fuc.SpatialClusterFlow at 0x1ce76762cd0>,
  'sfc003': <spatial_flow_clustering_fuc.SpatialClusterFlow at 0x1ce767535b0>,
  'sfc011': <spatial_flow_clustering_fuc.SpatialClusterFlow at 0x1ce0f6a15e0>,
  'sfc014': <spatial_flow_clustering_fuc.SpatialClusterFlow at 0x1ce0f6a1430>}}

In [4]:
selected_uid = '945a5b919ea73a83eeee7c**********'
sfc_obj = each_spatial_flow_cluster_dict[selected_uid]['sfc000']
print(sfc_obj.record_num)
sfc_obj.including_record_detail


10


{'od_0003': {'origin': [12691797.798597123, 2576646.3235157058],
  'destination': [12691800.657833291, 2577621.80660278],
  'start_time': '06:33:27',
  'end_time': '06:40:43',
  'date': '2021-07-12'},
 'od_0005': {'origin': [12691851.60042164, 2576664.527198888],
  'destination': [12691839.532645158, 2577627.4379340555],
  'start_time': '06:31:10',
  'end_time': '06:38:23',
  'date': '2021-07-13'},
 'od_0007': {'origin': [12691847.262155512, 2576673.3352038763],
  'destination': [12691816.343050309, 2577600.762768094],
  'start_time': '06:36:16',
  'end_time': '06:42:28',
  'date': '2021-07-14'},
 'od_0009': {'origin': [12691852.263567869, 2576661.877287062],
  'destination': [12691835.190458454, 2577664.9001906854],
  'start_time': '06:35:12',
  'end_time': '06:42:47',
  'date': '2021-07-15'},
 'od_0011': {'origin': [12691847.383585256, 2576666.9279766725],
  'destination': [12691835.736049842, 2577640.8776680985],
  'start_time': '06:40:43',
  'end_time': '06:48:19',
  'date': '2021-

In [7]:
selected_uid = '945a5b919ea73a83eeee7c**********'
sfc_obj = each_spatial_flow_cluster_dict[selected_uid]['fl000']
plot_sfc_obj(sfc_obj, selected_uid)

In [7]:
each_spatial_temporal_flow_cluster_dict = {}

from spatial_temporal_flow_clustering_fuc import SpatialTemporalFlowCluster


def init_bike_record_with_stfc_obj(_sfc_obj):
    _bike_record_dict = {}
    _init_temporal_spatial_flow_cluster_dict = {}
    for _num, (_uuid, _record_info) in enumerate(sfc_obj.including_record_detail.items()):
        _stfc = SpatialTemporalFlowCluster(_num,
                                           _sfc_obj.sfc_id,
                                           _sfc_obj.flow,
                                           _sfc_obj.record_num,
                                           _record_info['start_time'],
                                           _record_info['end_time'],
                                           {_uuid: _record_info})
        _init_temporal_spatial_flow_cluster_dict[_uuid] = _stfc
    return _init_temporal_spatial_flow_cluster_dict


def t1_is_earlier_than_t2(_t1, _t2):
    return 12 > (_t1 - _t2) >= 0 or (_t1 - _t2) < -12


def get_time_dist(_t1, _t2):
    return min(abs(_t1 - _t2), 24 - abs(_t1 - _t2))


# 0.5h = 30min
def calculate_temporal_similarity(_time_span1, _time_span2, _expansion_coefficient=0.5):
    _extended_time_span1 = [_time_span1[0] - _expansion_coefficient, _time_span1[1] + _expansion_coefficient]
    _extended_time_span2 = [_time_span2[0] - _expansion_coefficient, _time_span2[1] + _expansion_coefficient]

    if _extended_time_span1[0] - _expansion_coefficient < 0:
        _extended_time_span1[0] += 24
    if _extended_time_span1[1] + _expansion_coefficient >= 24:
        _extended_time_span1[1] -= 24
    if _extended_time_span2[0] - _expansion_coefficient < 0:
        _extended_time_span2[0] += 24
    if _extended_time_span2[1] + _expansion_coefficient >= 24:
        _extended_time_span2[1] -= 24
    if t1_is_earlier_than_t2(_extended_time_span1[0], _extended_time_span2[0]) is True and t1_is_earlier_than_t2(
            _extended_time_span1[1], _extended_time_span2[1]) is False:
        return 1 if get_time_dist(_extended_time_span1[0], _extended_time_span1[1]) < get_time_dist(
            _extended_time_span2[0],
            _extended_time_span2[1]) else 0
    elif t1_is_earlier_than_t2(_extended_time_span1[0], _extended_time_span2[0]) is False and t1_is_earlier_than_t2(
            _extended_time_span1[1], _extended_time_span2[1]) is True:
        return 1 if get_time_dist(_extended_time_span2[0], _extended_time_span2[1]) < get_time_dist(
            _extended_time_span1[0],
            _extended_time_span1[1]) else 0
    elif t1_is_earlier_than_t2(_extended_time_span1[1], _extended_time_span2[0]) is False or t1_is_earlier_than_t2(
            _extended_time_span2[1], _extended_time_span1[0]) is False:
        return 0
    else:
        _laser_flow_end_time = max(_extended_time_span1[1], _extended_time_span2[1])
        _earlier_flow_end_time = min(_extended_time_span1[1], _extended_time_span2[1])
        _laser_flow_start_time = max(_extended_time_span1[0], _extended_time_span2[0])
        _earlier_flow_start_time = min(_extended_time_span1[0], _extended_time_span2[0])
        return get_time_dist(_earlier_flow_end_time, _laser_flow_start_time) / get_time_dist(_earlier_flow_start_time,
                                                                                             _laser_flow_end_time)
    
    
calculate_temporal_similarity([6.5575, 6.6786111111111115], [6.519444444444445, 6.639722222222222])


0.9336208962377183

In [8]:

for uid, spatial_flow_cluster_dict in each_spatial_flow_cluster_dict.items():
    for stf_id, spatial_flow_cluster_obj in spatial_flow_cluster_dict.items():
        init_bike_record_with_stfc_dict = init_bike_record_with_stfc_obj(spatial_flow_cluster_obj)
        for uuid, stfc_obj in init_bike_record_with_stfc_dict.items():
            for another_uuid, another_stfc_obj in init_bike_record_with_stfc_dict.items():
                if uuid != another_uuid and stfc_obj.stfc_id != another_stfc_obj.stfc_id:
                    flows_ts = calculate_temporal_similarity(
                        stfc_obj.time_span,
                        another_stfc_obj.time_span, _expansion_coefficient = 0.5)
                    if flows_ts >= 0.5:
                        stfc_obj.add_flow(
                            another_stfc_obj.including_record_detail
                        )
                        init_bike_record_with_stfc_dict[another_uuid] = stfc_obj

[6.659166666666667, 6.659166666666667]
[6.675370370370371, 6.675370370370371]
[6.684791666666667, 6.684791666666667]
[6.708888888888889, 6.708888888888889]
[6.716990740740741, 6.716990740740741]
[6.719920634920634, 6.719920634920634]
[6.719791666666667, 6.719791666666667]
[15.175416666666667, 15.175416666666667]
[6.659166666666667, 6.659166666666667]
[6.675370370370371, 6.675370370370371]
[6.684791666666667, 6.684791666666667]
[6.708888888888889, 6.708888888888889]
[6.716990740740741, 6.716990740740741]
[6.719920634920634, 6.719920634920634]
[6.719791666666667, 6.719791666666667]
[15.175416666666667, 15.175416666666667]
[6.659166666666667, 6.659166666666667]
[6.675370370370371, 6.675370370370371]
[6.684791666666667, 6.684791666666667]
[6.708888888888889, 6.708888888888889]
[6.716990740740741, 6.716990740740741]
[6.719920634920634, 6.719920634920634]
[6.719791666666667, 6.719791666666667]
[15.175416666666667, 15.175416666666667]
[6.659166666666667, 6.659166666666667]
[6.675370370370371,

In [9]:
init_bike_record_with_stfc_dict['od_0003'].time_span

[6.719791666666667, 6.719791666666667]

In [13]:
init_bike_record_with_stfc_dict['od_0003'].record_start_time_list
sum(init_bike_record_with_stfc_dict['od_0003'].record_start_time_list) / len(
    init_bike_record_with_stfc_dict['od_0003'].record_start_time_list)

6.5905555555555555

In [12]:
sum(init_bike_record_with_stfc_dict['od_0003'].record_end_time_list) / len(init_bike_record_with_stfc_dict['od_0003'].record_end_time_list)

6.719791666666667